In [ ]:
import warnings
from pathlib import Path
import matplotlib.pyplot as plt

import prism

from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import (
    GenericMaterials,
    GenericStocks,
    MaterialIntensities,
    SharesInflowStocks
)
from imagematerials.vehicles.battery import ElectricVehicleBatteries, ElectricVehicleBatteries_TV, BatteryMaterials, EvBatteryLinkModule
from imagematerials.preprocessing import get_preprocessing_data
from imagematerials.electricity.preprocessing import get_preprocessing_data_evbattery, get_preprocessing_data_evbattery_old

from imagematerials.electricity.constants import STANDARD_SCEN_EXTERNAL_DATA

path_current = Path().resolve()
path_base = path_current.parent #.parent # base path of the project -> image-materials
path_data = Path(path_base, "data", "raw")

In [ ]:
vhc_sector = get_preprocessing_data("vehicles", Path("..", "data", "raw"), 
                                    climate_policy_scenario_dir = None, 
                                    circular_economy_scenario_dirs = None)


In [ ]:
path_data

In [ ]:
scenario = path_data /"electricity"/ STANDARD_SCEN_EXTERNAL_DATA
ev_battery_sector = get_preprocessing_data("ev_battery", path_data, 
                                    climate_policy_scenario_dir = Path(path_data, "image", "SSP2_M_CP"), 
                                    circular_economy_scenario_dirs = None,
                                    cache = False, 
                                    standard_scenario = scenario)

In [ ]:
prep_data = get_preprocessing_data_evbattery_old(path_base=path_base, scenario = "SSP2_M_CP", year_start=1926, year_out=2100)
sec_evbattery = Sector("ev_batteries", prep_data, check_coordinates=False)

In [ ]:
scenario_list = {"SSP2_M_CP":("SSP2_M_CP", None),
                # "SSP2_narrow":("SSP2_narrow", "narrow"),
                 }

# scenario_base_path = Path(path_base) / 'circular_economy_scenarios'

time_start = 1971
time_end = 2100
complete_timeline = prism.Timeline(time_start, time_end, 1)
simulation_timeline = prism.Timeline(time_start, time_end, 1)

all_output = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    climate_policy_scenario_dir = Path(path_data, "image", climate_scen)
    # climate_policy_scenario_dir = Path(path_base, "IMAGE_CircoMod", climate_scen)
    circular_economy_scenario_dir = None #Path(path_base, "circular_economy_scenarios", circular_scen)
    scenario = path_base /"electricity"/ STANDARD_SCEN_EXTERNAL_DATA

    vhc_sector = get_preprocessing_data("vehicles", path_data, 
                                    climate_policy_scenario_dir = climate_policy_scenario_dir, 
                                    circular_economy_scenario_dirs = None)
    ev_battery_sector = get_preprocessing_data("ev_battery", path_data,
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dir, cache = False, standard_scenario = scenario)     
    elc_sector = get_preprocessing_data("electricity", path_data,
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dir, cache = False, standard_scenario = scenario) 
    # elc_sector is a list of preprocessing data for each electricity subsector
    

    factory = ModelFactory(
        [vhc_sector, ev_battery_sector, *elc_sector], complete_timeline
        ).add(GenericStocks, ["vehicles",
                              "elc_gen",
                              "elc_grid_lines",
                              "elc_grid_add",
                              "elc_stor_phs"]
        # ).add(GenericMaterials, ["vehicles"]
        ).add(ElectricVehicleBatteries, ["ev_battery"], input_sources={
        "stock_by_cohort": "vehicles",
        "inflow": "vehicles",
        "outflow_by_cohort": "vehicles"}
        ).add(EvBatteryLinkModule, ["elc_stor_other"], input_sources={
        "stock_battery_kWh_v2g": "ev_battery"}
        ).add(SharesInflowStocks, ["elc_stor_other"],
        ).add(MaterialIntensities, ["elc_gen",
                                    "elc_grid_lines",
                                    "elc_grid_add",
                                    "elc_stor_phs",
                                    "elc_stor_other"
                                    ]
        )
    model = factory.finish()

    model.simulate(simulation_timeline)
    all_output[scen_id] = model
    print(f"Finished {scen_id}")

list(model.ev_battery)

In [ ]:
# Plot ev_battery: 1 variable over time
var = "inflow_battery_kWh"
m = model.ev_battery[var].to_array().sum(["Region", "BatteryType"])
m
for t in m.Type.values:
    plt.plot(m.time, m.sel(Type=t), label=str(t))
plt.xlabel("Year")
plt.ylabel(f"{var}")
plt.legend(bbox_to_anchor=(1.02, 0.5), loc="center left")
plt.title(f"{var}")